# Тестовый практикум: фильтрация батиметрических промеров

**Платформа FOLUR Казахстан · Этап 1 (прототип)** · связан с модулями А4 / П11.

> ⏱ **Трудозатраты:** ~15 мин.

Учимся базовому конвейеру обработки промеров глубин: загрузка → отбраковка выбросов → карта глубин.

⚠️ **Данные в этом ноутбуке — синтетические** (сгенерированы кодом ниже для демонстрации конвейера).
Реальный батиметрический датасет водоёмов Костанайской области публикуется на Zenodo (с DOI) на Этапе 2.


In [ ]:
# Шаг 1. Синтетический датасет промеров: имитация эхолотной съёмки озера
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # фиксируем зерно — результат воспроизводим
n = 500
x = rng.uniform(0, 1000, n)   # метры, локальная сетка (не реальные координаты)
y = rng.uniform(0, 600, n)
# 'чашеобразное' дно глубиной до ~8 м + шум измерения
depth = 8 * np.exp(-(((x-500)/350)**2 + ((y-300)/220)**2)) + rng.normal(0, 0.15, n)
# внесём 3% выбросов (сбои эхолота: ложное дно, пропуски)
bad = rng.choice(n, size=int(0.03*n), replace=False)
depth[bad] = rng.choice([0.0, 25.0, -1.0], size=bad.size)
df = pd.DataFrame({'x': x, 'y': y, 'depth_m': depth})
df.describe().round(2)


## Шаг 2. Отбраковка выбросов

Правила из модуля А4: физически невозможные значения (глубина ≤ 0 или больше максимально правдоподобной для водоёма) отбраковываются; далее — скользящий контроль по соседям.


In [ ]:
MAX_PLAUSIBLE = 12.0  # м, из паспорта водоёма / архивных съёмок
clean = df[(df.depth_m > 0) & (df.depth_m <= MAX_PLAUSIBLE)].copy()
print(f'Промеров: {len(df)} -> после фильтра: {len(clean)} (отбраковано {len(df)-len(clean)})')


## Шаг 3. Карта глубин (интерполяция на сетку)


In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

gx, gy = np.mgrid[0:1000:200j, 0:600:120j]
grid = griddata(clean[['x','y']].values, clean.depth_m.values, (gx, gy), method='linear')

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.contourf(gx, gy, grid, levels=12, cmap='Blues')
ax.scatter(clean.x, clean.y, s=3, c='k', alpha=0.25, label='промеры')
fig.colorbar(im, label='Глубина, м')
ax.set_title('Карта глубин (синтетическая демонстрация)')
ax.set_xlabel('x, м'); ax.set_ylabel('y, м'); ax.legend()
plt.show()


## Шаг 4. Верификация (обязательный шаг любого конвейера)

1. Сравните `clean.depth_m.max()` с паспортной глубиной водоёма — интерполяция не должна «углублять» озеро.
2. Визуально: изобаты замкнуты? нет ли «кратеров» вокруг одиночных точек (признак пропущенного выброса)?

!!! ИИ-задание (low-code): попросите LLM-ассистент Colab переписать Шаг 2 как функцию с параметрами
и проверьте, что результат фильтрации совпал с исходным (`len(clean)` не изменился).
Правило платформы: **ИИ ускоряет — человек верифицирует**.


In [ ]:
# Самопроверка (мини-тест практикума)
assert (clean.depth_m > 0).all() and (clean.depth_m <= MAX_PLAUSIBLE).all()
assert np.nanmax(grid) <= MAX_PLAUSIBLE + 1e-6
print('OK: конвейер прошёл самопроверку')
